<h1 style="font-size:20px; font-family: Verdana;">ALZHEIMER DESEASE BINARY CLASSIFICATION</h1>

In [1]:
import torch
print(f"Versione PyTorch: {torch.__version__}")
print(f"CUDA disponibile? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU rilevata: {torch.cuda.get_device_name(0)}")
else:
    print("ATTENZIONE: PyTorch non vede la GPU!")

Versione PyTorch: 2.5.1+cu121
CUDA disponibile? True
GPU rilevata: NVIDIA GeForce RTX 4090


1. LIBRARIES

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import torch.nn.functional as F
import nibabel as nib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random

2. DATASET AND CLASSES


In [3]:
#loading the dataset
with open("Dataset_Edin.pkl", "rb") as f:
    data_dict=pickle.load(f)

In [ ]:
#filtering class 0 e 2
#creating a mask to choose only the index where the label is 0 or 2

X_fa_raw=np.array(data_dict['FA'])
X_md_raw=np.array(data_dict['MD'])
y_raw=np.array(data_dict['Labels'])

#filtering class 0 e 2
#creating a mask to choose only the index where the label is 0 or 2
mask=(y_raw==0) | (y_raw==2)
X_fa=X_fa_raw[mask]
X_md=X_md_raw[mask]
y=y_raw[mask]

#trasforming class 2 in class 1 for binary classification
y[y==2]=1

print(f"Nuovo dataset: {len(y)} campioni")
print(f"Classe 0 Sani: {np.sum(y==0)} | Classe 1 (Alzheimer): {np.sum(y==1)}")

ANALISI_CORRENTE="FA"
X_input=X_fa if ANALISI_CORRENTE=="FA" else X_md
print(f"Analisi {ANALISI_CORRENTE} | Dataset: {len(y)} campioni")

Nuovo dataset: 317 campioni
Classe 0 Sani: 185 | Classe 1 (Alzheimer): 132


In [5]:
class AlzheimerDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths=paths
        self.labels=labels

    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        #carica il volume originale (FA o MD a seconda della lista passata)
        img=np.load(self.paths[idx])

        #trasforma il tensore[D,H,W]-->[1,D,H,W]
        img=torch.from_numpy(img).float().unsqueeze(0)

        #normalizzazione Min-Max per mantenere i valori tra 0 e 1
        if img.max() > img.min():
            img = (img - img.min())/(img.max()-img.min())

        label=torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label
    
    def collate_fn(batch):
        """
        Questa funzione serve a gestire volumi di dimensioni diverse nel batch
        facendo padding automatico alla dimensione massima del batch.
        """
        imgs, labels = zip(*batch)
    
         # Trova la dimensione massima in D, H, W
        max_d = max(img.shape[1] for img in imgs)
        max_h = max(img.shape[2] for img in imgs)
        max_w = max(img.shape[3] for img in imgs)
    
        padded_imgs = []
        for img in imgs:
            diff_d = max_d - img.shape[1]
            diff_h = max_h - img.shape[2]
            diff_w = max_w - img.shape[3]
        
            # Padding (sinistra, destra, sopra, sotto, davanti, dietro)
            img = F.pad(img, (0, diff_w, 0, diff_h, 0, diff_d))
            padded_imgs.append(img)
        
        return torch.stack(padded_imgs), torch.tensor(labels)

3. 3D CNN ARCHITECTURE


In [6]:
class AlzheimerCNN3D(nn.Module):
    def __init__(self, num_classes=2):
        super(AlzheimerCNN3D, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(2),
            
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(),
            nn.MaxPool3d(2),
            
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(),
            nn.MaxPool3d(2)
        )
        
        # Adaptive pooling: trasforma qualsiasi volume in [Batch, 128, 1, 1, 1]
        self.gap = nn.AdaptiveAvgPool3d(1)
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.6),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.LayerNorm(128),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x